# NexoUrbano · M3 — Ingesta Lakehouse (Databricks)
## ¿Qué armamos en este notebook?
Acá concentramos todo el flujo de ingesta y procesamiento de datos para nuestro proyecto:
1. Levantamos el dataset de viajes (capa Bronze) desde el almacenamiento.
2. Hacemos un perfilado completo para revisar nulos, tipos de datos y forma.
3. Calculamos las 5 métricas de negocio clave del sistema.
4. Consultamos la API pública de **Open-Meteo** para las estaciones meteorológicas.
5. Integramos las menciones de redes sociales (usando Bluesky y un fallback a Mastodon).
6. Consolidamos y guardamos todo limpio en la capa **Silver** lista para consumir.

## Características clave
- **Reejecutable:** Centralizamos todos los parámetros al inicio mediante widgets. No dejamos rutas ni fechas hardcodeadas en medio del código.
- **Sin secretos expuestos:** Ninguna de las APIs que consultamos requiere claves ocultas. Si el equipo llega a utilizar credenciales privadas, se manejan externamente y nunca se suben al repositorio.

## 0 · Parámetros (widgets)

In [0]:

try:
    dbutils.widgets.text("ciudad", "Mendoza", "Ciudad")
    dbutils.widgets.text("fecha_inicio", "2026-03-01", "Fecha inicio (YYYY-MM-DD)")
    dbutils.widgets.text("fecha_fin", "2026-03-31", "Fecha fin (YYYY-MM-DD)")
    dbutils.widgets.text("tz", "America/Argentina/Mendoza", "Timezone IANA")
    dbutils.widgets.text("path_viajes", "dbfs:/FileStore/nexourbano/bronze/viajes/", "Path CSV viajes (Bronze)")
    dbutils.widgets.text("path_silver", "dbfs:/FileStore/nexourbano/silver/", "Path Silver")
    dbutils.widgets.text("social_query", "scooter electrico", "Query social")
    dbutils.widgets.text("social_hashtag", "movilidad", "Hashtag Mastodon (fallback)")
    dbutils.widgets.dropdown("formato_salida", "delta", ["delta", "parquet"], "Formato Silver")

    P = {k: dbutils.widgets.get(k) for k in [
        "ciudad", "fecha_inicio", "fecha_fin", "tz", "path_viajes",
        "path_silver", "social_query", "social_hashtag", "formato_salida"]}
except NameError:
    P = {
        "ciudad": "Mendoza",
        "fecha_inicio": "2025-03-01",
        "fecha_fin": "2025-03-31",  
        "tz": "America/Argentina/Mendoza",
        "path_viajes": "data/viajes.csv",
        "path_silver": "data/silver/",
        "social_query": "scooter electrico",
        "social_hashtag": "movilidad",
        "formato_salida": "parquet",
    }

# Estaciones de Mendoza
ESTACIONES = [
    {"station_id": "EST-001", "nombre": "Plaza Independencia", "lat": -32.8908, "lon": -68.8447},
    {"station_id": "EST-002", "nombre": "Parque General San Martin", "lat": -32.8900, "lon": -68.8760},
    {"station_id": "EST-003", "nombre": "Terminal de Omnibus", "lat": -32.8861, "lon": -68.8280},
]

print("Parámetros de esta corrida:")
for k, v in P.items():
    print(f"  {k:16s} = {v}")

Parámetros de esta corrida:
  ciudad           = Mendoza
  fecha_inicio     = 2025-03-01
  fecha_fin        = 2025-03-31
  tz               = America/Argentina/Mendoza
  path_viajes      = abfss://bronze@nexourbanodata2026.dfs.core.windows.net/viajes/2026/09/16/viajes.csv
  path_silver      = abfss://silver@nexourbanodata2026.dfs.core.windows.net/
  social_query     = scooter electrico
  social_hashtag   = movilidad
  formato_salida   = delta


## 1 · Setup

In [0]:
import json
import re
import time
import urllib.parse
import urllib.request
from datetime import date, datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql import types as T

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("nexourbano-m3").getOrCreate()

spark.conf.set("spark.sql.session.timeZone", P["tz"])

USER_AGENT = "NexoUrbano-TP-2026/1.0 (uso academico)"


def http_get_json(url, timeout=30, reintentos=3, pausa=2):
    """GET con reintento exponencial. Devuelve (json, None) o (None, motivo_error)."""
    ultimo_error = None
    for intento in range(1, reintentos + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT,
                                                       "Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("utf-8")), None
        except Exception as e:  # HTTPError, URLError, JSONDecodeError
            ultimo_error = f"{type(e).__name__}: {e}"
            if intento < reintentos:
                time.sleep(pausa ** intento)
    return None, ultimo_error


def escribir_silver(df, nombre_tabla):
    """Escribe en Silver en delta o parquet según el widget. Idempotente (overwrite)."""
    ruta = P["path_silver"].rstrip("/") + "/" + nombre_tabla
    fmt = P["formato_salida"]
    try:
        (df.write.format(fmt).mode("overwrite")
           .option("overwriteSchema", "true").save(ruta))
    except Exception as e:
        print(f"[WARN] Falló escritura en {fmt} ({type(e).__name__}). Degradando a parquet.")
        fmt = "parquet"
        df.write.format("parquet").mode("overwrite").save(ruta)
    print(f"[OK] {nombre_tabla} -> {ruta} (formato={fmt}, filas={df.count()})")
    return ruta

## 1.1 · Conexión al Storage de Azure (ADLS)

 Por regla del proyecto, **jamás dejamos keys hardcodeadas en el código**. La clave de acceso se ingresa de forma segura a través del widget interactivo de la sesión y no se persiste al exportar el notebook. Si el entorno corre localmente o usa otra ruta, la celda lo detecta automáticamente y avanza sin problemas.

In [0]:
try:
    import azure.storage.blob  # noqa: F401
except ImportError:
    %pip install --quiet azure-storage-blob
    import azure.storage.blob  # noqa: F401

import re as _re

try:
    dbutils.widgets.text("storage_account", "nexourbanodata2026", "Storage account (ADLS)")
    dbutils.widgets.text("storage_key", "", "Access Key (pegar a mano, no commitear)")
    storage_account = dbutils.widgets.get("storage_account")
    storage_key = dbutils.widgets.get("storage_key")
except NameError:
    storage_account, storage_key = "", ""


def resolver_path_viajes(path_original, storage_account, storage_key):
    if not path_original.startswith("abfss://"):
        print("[SKIP] path_viajes no es abfss://, no hace falta credencial.")
        return path_original

    if not storage_key:
        print("[ATENCION] path_viajes es abfss:// pero storage_key está vacío. "
              "Pegar la Access Key en el widget y volver a correr esta celda.")
        return path_original

    m = _re.match(r"abfss://([^@]+)@([^.]+)\.dfs\.core\.windows\.net/(.+)", path_original)
    if not m:
        print(f"[ERROR] No se pudo parsear la URI abfss: {path_original}")
        return path_original
    contenedor, cuenta, blob_path = m.groups()

    from azure.storage.blob import BlobServiceClient

    conn_str = (f"DefaultEndpointsProtocol=https;AccountName={cuenta};"
                f"AccountKey={storage_key};EndpointSuffix=core.windows.net")
    try:
        cliente = BlobServiceClient.from_connection_string(conn_str)
        blob = cliente.get_blob_client(container=contenedor, blob=blob_path)
        nombre_archivo = blob_path.rsplit("/", 1)[-1]
        destino_local = f"/tmp/{nombre_archivo}"
        with open(destino_local, "wb") as f:
            f.write(blob.download_blob().readall())
        print(f"[OK] Descargado {contenedor}/{blob_path} -> {destino_local}")
        return f"file:{destino_local}"
    except Exception as e:
        print(f"[ERROR] Falló la descarga desde Azure: {type(e).__name__}: {e}")
        return path_original


path_viajes_resuelto = resolver_path_viajes(P["path_viajes"], storage_account, storage_key)
P["path_viajes"] = path_viajes_resuelto
print(f"path_viajes final -> {P['path_viajes']}")


[SKIP] path_viajes no es abfss://, no hace falta credencial.
path_viajes final -> file:/tmp/viajes.csv


## 2 · Bronze: lectura del CSV de viajes

El TP pide un dataset abierto real (Citi Bike, EcoBici, GBFS) o el generador.
Cada fuente trae nombres de columna distintos, así que abajo hay un
**mapeo explícito a un esquema canónico**. Eso es parte del contrato Bronze (M1).

In [0]:
if P["path_viajes"].startswith("file:"):
    import pandas as pd
    local_path = P["path_viajes"].replace("file:", "", 1)
    df_raw = spark.createDataFrame(pd.read_csv(local_path))
else:
    df_raw = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .option("mode", "PERMISSIVE")
              .csv(P["path_viajes"]))

print(f"Columnas originales del origen ({len(df_raw.columns)}):")
print(df_raw.columns)
df_raw.printSchema()

Columnas originales del origen (17):
['ride_id', 'ts_start', 'ts_end', 'duration_s', 'rider_id', 'vehicle_id', 'station_start', 'station_end', 'barrio_start', 'barrio_end', 'lat_start', 'lon_start', 'lat_end', 'lon_end', 'km', 'fare_ars', 'clima_codigo']
root
 |-- ride_id: string (nullable = true)
 |-- ts_start: string (nullable = true)
 |-- ts_end: string (nullable = true)
 |-- duration_s: long (nullable = true)
 |-- rider_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- station_start: string (nullable = true)
 |-- station_end: string (nullable = true)
 |-- barrio_start: string (nullable = true)
 |-- barrio_end: string (nullable = true)
 |-- lat_start: double (nullable = true)
 |-- lon_start: double (nullable = true)
 |-- lat_end: double (nullable = true)
 |-- lon_end: double (nullable = true)
 |-- km: double (nullable = true)
 |-- fare_ars: double (nullable = true)
 |-- clima_codigo: long (nullable = true)



In [0]:
# Mapeo origen -> esquema canónico NexoUrbano.

CANDIDATOS = {
    "ts_inicio":  ["ts_inicio", "ts_start", "started_at", "starttime", "start_time",
                   "fecha_origen_recorrido", "start_date"],
    "ts_fin":     ["ts_fin", "ts_end", "ended_at", "stoptime", "end_time",
                   "fecha_destino_recorrido", "end_date"],
    "estacion_origen": ["station_start", "station_id_origen", "start_station_id",
                        "start_station_name", "id_estacion_origen", "nombre_estacion_origen"],
    "estacion_destino": ["station_end", "station_id_destino", "end_station_id",
                         "end_station_name", "id_estacion_destino", "nombre_estacion_destino"],
    "duracion_s": ["duration_s", "duracion_recorrido", "tripduration", "duration_sec"],
    "rider_id":   ["rider_id", "user_id", "id_usuario", "member_id"],
    "lat_origen": ["lat_start", "lat_origen", "start_lat", "start_station_latitude", "lat"],
    "lon_origen": ["lon_start", "lon_origen", "start_lng", "start_station_longitude", "lon", "lng"],
}

cols_lower = {c.lower(): c for c in df_raw.columns}
mapeo, faltantes = {}, []
for canonico, opciones in CANDIDATOS.items():
    hallado = next((cols_lower[o] for o in opciones if o in cols_lower), None)
    if hallado:
        mapeo[canonico] = hallado
    else:
        faltantes.append(canonico)

print("Mapeo resuelto:")
for k, v in mapeo.items():
    print(f"  {k:18s} <- {v}")
if faltantes:
    print(f"\n[ATENCION] Sin mapear: {faltantes}")
    print("  -> agregar el nombre real a CANDIDATOS antes de seguir.")

assert "ts_inicio" in mapeo, "No se encontró la columna de timestamp de inicio. Editar CANDIDATOS."

Mapeo resuelto:
  ts_inicio          <- ts_start
  ts_fin             <- ts_end
  estacion_origen    <- station_start
  estacion_destino   <- station_end
  duracion_s         <- duration_s
  rider_id           <- rider_id
  lat_origen         <- lat_start
  lon_origen         <- lon_start


In [0]:
print(list(df_raw.columns))

['ride_id', 'ts_start', 'ts_end', 'duration_s', 'rider_id', 'vehicle_id', 'station_start', 'station_end', 'barrio_start', 'barrio_end', 'lat_start', 'lon_start', 'lat_end', 'lon_end', 'km', 'fare_ars', 'clima_codigo']


In [0]:
ruta_hmv = "/tmp/silver_viajes_clima"
fmt = "parquet"

try:
    if 'df_viajes_clima' in locals():
        (df_viajes_clima.write
         .format(fmt)
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .save(ruta_hmv))
        print(f"[OK] Tabla HMV guardada exitosamente en local en: {ruta_hmv}")
    else:
        print("[ERROR] La variable 'df_viajes_clima' no está definida.")
except Exception as e:
    print(f"[WARN] Error al guardar: {e}")

[WARN] Error al guardar: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /tmp/silver_viajes_clima/_delta_log SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.getFileStatus(DatabricksFileSystemV2.scala:752)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.getFileStatus(DatabricksFileSystem.scala:215)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$4(LokiFileSystem.scala:707)
	at com.databricks.sql.io.LokiFileSystem.tryWithNativeIO(LokiFileSystem.scala:679)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$1(LokiFileSystem.scala:707)
	at com.databricks.sql.io.LokiFileSystem$.withWrappedExceptions(LokiFileSystem.scala:181)
	at com.databricks.sql.io.LokiFileSystem.getFil

## 3 · Perfilado (shape, nulos, tipos, cardinalidad)

Esto alimenta el contrato Bronze del M1. Lo versionamos, no lo screenshoteamos.

In [0]:
# Aseguramos que df_viajes esté definido en memoria
df_viajes = df_raw

n_filas = df_viajes.count()
n_cols = len(df_viajes.columns)
print(f"SHAPE: {n_filas:,} filas x {n_cols} columnas")

perfil = []
for c, tipo in df_viajes.dtypes:
    fila = df_viajes.agg(
        F.count(F.when(F.col(c).isNull(), c)).alias("nulos"),
        F.approx_count_distinct(c).alias("distintos"),
    ).collect()[0]
    perfil.append({
        "columna": c,
        "tipo": tipo,
        "nulos": int(fila["nulos"]),
        "distintos": int(fila["distintos"])
    })

SHAPE: 250,000 filas x 17 columnas


In [0]:
# Definimos df_viajes renombrando las columnas del origen al formato canónico
df_viajes = df_raw.withColumnRenamed("ts_start", "ts_inicio") \
                  .withColumnRenamed("ts_end", "ts_fin") \
                  .withColumnRenamed("station_start", "estacion_origen") \
                  .withColumnRenamed("station_end", "estacion_destino") \
                  .withColumnRenamed("duration_s", "duracion_s")


resumen = df_viajes.select(
    F.min("ts_inicio").alias("ts_min"),
    F.max("ts_inicio").alias("ts_max"),
    F.min("duracion_s").alias("dur_min"),
    F.max("duracion_s").alias("dur_max"),
).collect()[0]

print(f"Ventana temporal del dataset : {resumen['ts_min']} -> {resumen['ts_max']}")
print(f"Duración min/max (s)         : {resumen['dur_min']} / {resumen['dur_max']}")

Ventana temporal del dataset : 2026-03-01T00:00:00+00:00 -> 2026-05-30T00:00:00+00:00
Duración min/max (s)         : 8 / 20000


## 4 · Las 5 métricas de negocio

In [0]:
df_viajes_ok = df_viajes.filter(
    F.col("ts_inicio").isNotNull() &
    F.col("duracion_s").between(30, 4 * 3600)
)
print(f"Silver viajes: {df_viajes_ok.count():,} filas "
      f"({100.0 * df_viajes_ok.count() / n_filas:.1f}% del Bronze)")

Silver viajes: 244,955 filas (98.0% del Bronze)


In [0]:
import pyspark.sql.functions as F

m1 = (df_viajes_ok
      .withColumn("hora", F.hour(F.col("ts_inicio")))
      .groupBy("hora")
      .agg(F.count("*").alias("viajes"))
      .orderBy("hora"))

display(m1) if "display" in dir() else m1.show(24)

hora,viajes
0,10290
1,10257
2,10073
3,10249
4,10153
5,10289
6,10249
7,10236
8,10185
9,10227


In [0]:
# M2 · duración p50 / p95 (en minutos)
m2 = df_viajes_ok.select(
    F.expr("percentile_approx(duracion_s, 0.50)").alias("p50_s"),
    F.expr("percentile_approx(duracion_s, 0.95)").alias("p95_s"),
    F.avg("duracion_s").alias("media_s"),
).withColumn("p50_min", F.round(F.col("p50_s") / 60, 1)) \
 .withColumn("p95_min", F.round(F.col("p95_s") / 60, 1))
display(m2) if "display" in dir() else m2.show()

p50_s,p95_s,media_s,p50_min,p95_min
244,771,312.02555571431486,4.1,12.9


In [0]:
# M3 · top 10 estaciones de origen
if "estacion_origen" in df_viajes_ok.columns:
    m3 = (df_viajes_ok.groupBy("estacion_origen")
          .agg(F.count("*").alias("viajes"),
               F.round(F.avg("duracion_s") / 60, 1).alias("dur_media_min"))
          .orderBy(F.desc("viajes")).limit(10))
    display(m3) if "display" in dir() else m3.show(truncate=False)
else:
    print("[SKIP] El dataset no trae estación de origen.")

estacion_origen,viajes,dur_media_min
ST019,10357,5.2
ST020,10340,5.2
ST009,10337,5.2
ST012,10321,5.2
ST003,10313,5.1
ST007,10298,5.2
ST017,10280,5.2
ST005,10250,5.2
ST010,10233,5.2
ST021,10226,5.2


In [0]:
import pyspark.sql.functions as F
df_temp = df_viajes_ok.withColumn("fecha", F.to_date(F.col("ts_inicio"))) \
                      .withColumn("dia_semana", F.date_format(F.col("ts_inicio"), "EEEE"))

# 2. M4 · viajes por día + estacionalidad semanal
m4 = (df_temp.groupBy("fecha")
      .agg(F.count("*").alias("viajes"))
      .orderBy("fecha"))

display(m4) if "display" in dir() else m4.show(31)

m4b = (df_temp.groupBy("dia_semana")
       .agg(F.count("*").alias("viajes"))
       .orderBy(F.desc("viajes")))

display(m4b) if "display" in dir() else m4b.show()

fecha,viajes
2026-03-01,2689
2026-03-02,2697
2026-03-03,2759
2026-03-04,2703
2026-03-05,2681
2026-03-06,2709
2026-03-07,2738
2026-03-08,2695
2026-03-09,2679
2026-03-10,2762


dia_semana,viajes
Monday,35518
Wednesday,35491
Tuesday,35472
Thursday,35436
Sunday,35215
Friday,35068
Saturday,32755


La métrica 5 (**% de viajes con clima adverso**) se calcula recién después del
join con Open-Meteo, en la sección 7.

## 5 · API de clima — Open-Meteo (sin API key)

Se elige endpoint según la fecha pedida:
- `archive-api` (ERA5) para fechas con más de ~5 días de antigüedad.
- `api` (forecast) para fechas recientes o futuras.

In [0]:
def url_open_meteo(lat, lon, desde, hasta, tz):
    hourly = "temperature_2m,precipitation,wind_speed_10m,weather_code"
    es_historico = datetime.strptime(hasta, "%Y-%m-%d").date() < date.today() - timedelta(days=6)
    base = ("https://archive-api.open-meteo.com/v1/archive" if es_historico
            else "https://api.open-meteo.com/v1/forecast")
    q = urllib.parse.urlencode({
        "latitude": lat, "longitude": lon,
        "start_date": desde, "end_date": hasta,
        "hourly": hourly, "timezone": tz,
    })
    return f"{base}?{q}", ("archive" if es_historico else "forecast")


registros_clima, errores_clima = [], []

for est in ESTACIONES:
    url, endpoint = url_open_meteo(est["lat"], est["lon"],
                                   P["fecha_inicio"], P["fecha_fin"], P["tz"])
    print(f"[{est['station_id']}] {endpoint} -> {url[:110]}...")
    data, err = http_get_json(url)
    if err:
        errores_clima.append((est["station_id"], err))
        print(f"  [ERROR] {err}")
        continue

    h = data.get("hourly", {})
    for i, ts in enumerate(h.get("time", [])):
        registros_clima.append({
            "station_id": est["station_id"],
            "estacion_nombre": est["nombre"],
            "lat": float(est["lat"]),
            "lon": float(est["lon"]),
            "ts": datetime.fromisoformat(ts),
            "temp_c": h["temperature_2m"][i],
            "precip_mm": h["precipitation"][i],
            "viento_kmh": h["wind_speed_10m"][i],
            "weather_code": h["weather_code"][i],
            "fuente": f"open-meteo/{endpoint}",
        })
    time.sleep(1)  # cortesía con la API pública

print(f"\nRegistros horarios de clima: {len(registros_clima):,}")
if errores_clima:
    print(f"Estaciones con error: {errores_clima}")

assert registros_clima, "Open-Meteo no devolvió datos. Revisar fechas/coordenadas."

[EST-001] archive -> https://archive-api.open-meteo.com/v1/archive?latitude=-32.8908&longitude=-68.8447&start_date=2025-03-01&end_d...
[EST-002] archive -> https://archive-api.open-meteo.com/v1/archive?latitude=-32.89&longitude=-68.876&start_date=2025-03-01&end_date...
[EST-003] archive -> https://archive-api.open-meteo.com/v1/archive?latitude=-32.8861&longitude=-68.828&start_date=2025-03-01&end_da...

Registros horarios de clima: 2,232


In [0]:
schema_clima = T.StructType([
    T.StructField("station_id", T.StringType()),
    T.StructField("estacion_nombre", T.StringType()),
    T.StructField("lat", T.DoubleType()),
    T.StructField("lon", T.DoubleType()),
    T.StructField("ts", T.TimestampType()),
    T.StructField("temp_c", T.DoubleType()),
    T.StructField("precip_mm", T.DoubleType()),
    T.StructField("viento_kmh", T.DoubleType()),
    T.StructField("weather_code", T.IntegerType()),
    T.StructField("fuente", T.StringType()),
])

ruta_clima = P["path_silver"].rstrip("/") + "/silver_clima_horario"

try:
    df_clima.write.format("parquet").mode("overwrite").save(ruta_clima)
    print(f"[OK] Clima guardado exitosamente en: {ruta_clima}")
except Exception as e:
    print(f"[WARN] No se pudo guardar en Parquet: {e}")


[WARN] No se pudo guardar en Parquet: (com.databricks.common.filesystem.SparkKeyProviderException) Failure to initialize configuration for storage account nexourbanodata2026.dfs.core.windows.net: Invalid configuration value detected for fs.azure.account.key

JVM stacktrace:
com.databricks.common.filesystem.SparkKeyProviderException
	at com.databricks.common.filesystem.LokiABFS.initialize(LokiABFS.scala:48)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$3(LokiFileSystem.scala:137)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$2(LokiFileSystem.scala:133)
	at com.databricks.sql.io.FileSystemCache.getOrCompute(FileSystemCache.scala:60)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getL

## 6 · API social — Bluesky con fallback a Mastodon

**Nota de arquitectura (va al ADR):** el AppView público de Bluesky
(`public.api.bsky.app`) sirve lecturas sin auth, pero `searchPosts` empezó a
responder 403 a llamadas anónimas de forma intermitente. Por eso el conector
tiene fallback a la API pública de Mastodon (timeline por hashtag), que no
requiere credenciales. Ambos producen el **mismo esquema canónico**, así que
la tabla Silver no cambia según quién respondió. Esto es "veracidad" aplicada:
la fuente queda registrada en la columna `red`.

In [0]:
def limpiar_html(s):
    if not s:
        return ""
    s = re.sub(r"<br\s*/?>", " ", s)
    s = re.sub(r"</p>", " ", s)
    s = re.sub(r"<[^>]+>", "", s)
    return re.sub(r"\s+", " ", s).strip()


def traer_bluesky(query, limite=50):
    url = ("https://public.api.bsky.app/xrpc/app.bsky.feed.searchPosts?"
           + urllib.parse.urlencode({"q": query, "limit": min(limite, 100)}))
    data, err = http_get_json(url, reintentos=2)
    if err:
        return None, err
    posts = []
    for p in data.get("posts", []):
        rec = p.get("record", {})
        posts.append({
            "red": "bluesky",
            "post_id": p.get("uri"),
            "autor": (p.get("author") or {}).get("handle"),
            "texto": rec.get("text", ""),
            "ts": rec.get("createdAt"),
            "likes": int(p.get("likeCount") or 0),
            "replies": int(p.get("replyCount") or 0),
            "idioma": ((rec.get("langs") or [None])[0]),
        })
    return posts, None


def traer_mastodon(hashtag, limite=40, instancia="https://mastodon.social"):
    url = (f"{instancia}/api/v1/timelines/tag/{urllib.parse.quote(hashtag)}?"
           + urllib.parse.urlencode({"limit": min(limite, 40)}))
    data, err = http_get_json(url, reintentos=2)
    if err:
        return None, err
    posts = []
    for p in data:
        posts.append({
            "red": "mastodon",
            "post_id": str(p.get("id")),
            "autor": (p.get("account") or {}).get("acct"),
            "texto": limpiar_html(p.get("content")),
            "ts": p.get("created_at"),
            "likes": int(p.get("favourites_count") or 0),
            "replies": int(p.get("replies_count") or 0),
            "idioma": p.get("language"),
        })
    return posts, None


posts, motivo = traer_bluesky(P["social_query"])
fuente_usada = "bluesky"

if not posts:
    print(f"[FALLBACK] Bluesky no respondió ({motivo}). Yendo a Mastodon.")
    posts, motivo = traer_mastodon(P["social_hashtag"])
    fuente_usada = "mastodon"

assert posts, f"Ninguna de las dos APIs sociales respondió. Último error: {motivo}"
print(f"[OK] Fuente social usada: {fuente_usada} · posts obtenidos: {len(posts)}")

[FALLBACK] Bluesky no respondió (HTTPError: HTTP Error 403: Forbidden). Yendo a Mastodon.
[OK] Fuente social usada: mastodon · posts obtenidos: 40


In [0]:
schema_social = T.StructType([
    T.StructField("red", T.StringType()),
    T.StructField("post_id", T.StringType()),
    T.StructField("autor", T.StringType()),
    T.StructField("texto", T.StringType()),
    T.StructField("ts", T.StringType()),
    T.StructField("likes", T.IntegerType()),
    T.StructField("replies", T.IntegerType()),
    T.StructField("idioma", T.StringType()),
])

TERMINOS_OPS = ["bateria", "batería", "freno", "app", "cobro", "cobraron",
                "estacion", "estación", "accidente", "robo", "carga"]

df_social = (spark.createDataFrame(posts, schema=schema_social)
             .withColumn("ts", F.to_timestamp("ts"))
             .withColumn("fecha", F.to_date("ts"))
             .withColumn("hora", F.hour("ts"))
             .withColumn("texto_norm", F.lower(F.col("texto")))
             .withColumn("menciona_incidente",
                         F.arrays_overlap(
                             F.split(F.regexp_replace(F.lower(F.col("texto")), r"[^\wáéíóúñ ]", " "), r"\s+"),
                             F.array(*[F.lit(t) for t in TERMINOS_OPS])))
             .withColumn("query", F.lit(P["social_query"] if fuente_usada == "bluesky"
                                        else P["social_hashtag"]))
             .withColumn("ingest_ts", F.current_timestamp())
             .dropDuplicates(["red", "post_id"]))

display(df_social.select("red", "autor", "texto", "ts", "likes", "menciona_incidente").limit(15)) \
    if "display" in dir() else df_social.show(15, truncate=80)

print("Posts que mencionan un término operativo:",
      df_social.filter("menciona_incidente").count())

ruta_social = P["path_silver"].rstrip("/") + "/silver_social_menciones"

try:
    df_social.write.format("parquet").mode("overwrite").option("overwriteSchema", "true").save(ruta_social)
    print(f"[OK] Datos sociales guardados exitosamente en: {ruta_social}")
except Exception as e:
    print(f"[WARN] No se pudo guardar: {e}")

df_clima = spark.createDataFrame(registros_clima)

red autor texto ts likes menciona_incidente mastodon aidoo@masto.es Nuevo Reglamento DGT (1/10/2026): VMP desde 15 años con casco y luces (200€), motoristas con guantes y calzado cerrado, y al adelantar ciclistas: cambiar de carril, 1,5 m y -20 km/h. https://aidoo.news/noticia/r9P5no #España #Movilidad #Patinetes #Ciclismo #Motociclismo 2026-09-24T10:06:10.000-03:00 0 false mastodon aidoo Nuevo Reglamento DGT (1/10/2026): VMP desde 15 años con casco y luces (200€), motoristas con guantes y calzado cerrado, y al adelantar ciclistas: cambiar de carril, 1,5 m y -20 km/h. https://aidoo.news/noticia/r9P5no #España #Movilidad #Patinetes #Ciclismo #Motociclismo 2026-09-24T10:06:10.369-03:00 0 false mastodon tsumy@tardigram.com 2026-09-24T07:35:18.000-03:00 1 false mastodon sociedad-noticias.com@sociedad-noticias.com Uber Teens incorpora video en vivo para supervisar viajes de adolescentes Uber habilitó en México video en vivo para Uber Teens, permitiendo a padres y tutores observar el interior del vehículo durante el viaje de adolescentes. Por Deyanira Vázquez | Reportera                                         Uber incorporó en México una función de video en vivo para los viajes de Uber Teens, con la que padres y tutores podrán observar desde su propia aplicación lo que ocurre dentro del vehículo mientras un adolescente realiza el trayecto. La nueva herramienta está disponible desde esta semana en las ciudades del país donde opera la plataforma, aunque su activación depende de las condiciones técnicas del teléfono del conductor. Cuando esté disponible, el tutor recibirá una notificación al comenzar el viaje. La transmisión es captada mediante el teléfono del conductor y únicamente puede ser consultada por el tutor vinculado a la cuenta del adolescente. Al concluir el recorrido, el acceso al video queda restringido y el archivo permanece protegido, de acuerdo con la información proporcionada por la empresa. Así funciona el video en vivo de Uber Teens Uber Teens permite que adolescentes de entre 13 y 17 años soliciten viajes bajo supervisión de sus padres o tutores mediante un Perfil Familiar. El servicio fue habilitado en México en junio de 2024 y cuenta con herramientas de seguimiento y seguridad integradas. Con la nueva función, el tutor podrá recibir una transmisión del interior del vehículo durante el recorrido, además de consultar información que ya estaba disponible en Uber Teens, como el nombre del conductor, placas, marca y modelo del automóvil y destino solicitado. La empresa precisó que el video en vivo no estará disponible necesariamente en todos los viajes. Su funcionamiento depende de que el dispositivo del conductor reúna las condiciones técnicas necesarias para realizar la grabación. Cecilia Roman, gerente de Comunicación de Seguridad para Uber en México, señaló que la compañía amplió las herramientas del Perfil Familiar con el objetivo de ofrecer una opción adicional de supervisión durante los traslados de adolescentes. Seguridad y supervisión parental Uber Teens mantiene otras herramientas de seguridad durante los viajes. Entre ellas se encuentra el seguimiento en tiempo real, mediante el cual los titulares del Perfil Familiar pueden conocer el avance del recorrido desde que el adolescente solicita el servicio. También está disponible la comunicación anonimizada entre padres o tutores y el conductor, así como un chat grupal para coordinar la recogida del adolescente sin necesidad de realizar llamadas telefónicas. Otra medida es la verificación mediante código PIN. Antes de abordar, el adolescente recibe un código que debe proporcionar al conductor; el viaje solamente puede comenzar cuando el código correcto es ingresado en la aplicación del conductor. ¿Qué ocurre si el viaje presenta una anomalía? Uber Teens también utiliza RideCheck, una herramienta basada en tecnología GPS que puede detectar situaciones como desviaciones inusuales de la ruta, detenciones inesperadas o la finalización anticipada de un viaje. Cuando 

Posts que mencionan un término operativo: 2
[WARN] No se pudo guardar: (com.databricks.common.filesystem.SparkKeyProviderException) Failure to initialize configuration for storage account nexourbanodata2026.dfs.core.windows.net: Invalid configuration value detected for fs.azure.account.key

JVM stacktrace:
com.databricks.common.filesystem.SparkKeyProviderException
	at com.databricks.common.filesystem.LokiABFS.initialize(LokiABFS.scala:48)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$3(LokiFileSystem.scala:137)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$2(LokiFileSystem.scala:133)
	at com.databricks.sql.io.FileSystemCache.getOrCompute(FileSystemCache.scala:60)
	at com.databricks.sql

## 7 · Tabla Silver del HMV: `silver_viajes_clima`

**Grano:** una fila por viaje. **Clave:** `(estacion_origen, ts_inicio, rider_id)`.
**Join:** viaje ⨝ clima por `(station_id, fecha, hora)`.

Si las estaciones del dataset abierto no coinciden con las 3 de `ESTACIONES`,
se usa el clima de la ciudad (promedio de las 3) como fallback, y se marca en
`match_clima` cuál de los dos caminos se usó. Esto también es linaje.

In [0]:
ids_estaciones = [e["station_id"] for e in ESTACIONES]
hay_match_directo = False

if "estacion_origen" in df_viajes_ok.columns:
    hay_match_directo = (df_viajes_ok
                         .filter(F.col("estacion_origen").cast("string").isin(ids_estaciones))
                         .limit(1).count() > 0)

print(f"¿Los station_id del dataset matchean con ESTACIONES? -> {hay_match_directo}")

if hay_match_directo:
    df_clima_prep = df_clima.withColumn("fecha", F.to_date(F.col("ts"))) \
                            .withColumn("hora", F.hour(F.col("ts")))
    
    df_clima_join = df_clima_prep.select(
        "station_id", "fecha", "hora", "temp_c",
        "precip_mm", "viento_kmh", "weather_code",
        "clima_adverso", "fuente"
    )
    
    df_viajes_clima = (df_viajes_ok.alias("v")
                       .join(df_clima_join.alias("c"), 
                             (F.col("v.estacion_origen") == F.col("c.station_id")) & 
                             (F.to_date(F.col("v.ts_start")) == F.col("c.fecha")) & 
                             (F.hour(F.col("v.ts_start")) == F.col("c.hora")),
                             "left"))
else:
    df_viajes_clima = df_viajes_ok

¿Los station_id del dataset matchean con ESTACIONES? -> False


In [0]:
ruta_hmv = P["path_silver"].rstrip("/") + "/silver_viajes_clima"

try:
    df_viajes_clima.write.format("parquet").mode("overwrite").option("overwriteSchema", "true").save(ruta_hmv)
    print(f"[OK] Tabla HMV guardada exitosamente en Parquet en: {ruta_hmv}")
except Exception as e:
    print(f"[WARN] Error al guardar: {e}")

[WARN] Error al guardar: (com.databricks.common.filesystem.SparkKeyProviderException) Failure to initialize configuration for storage account nexourbanodata2026.dfs.core.windows.net: Invalid configuration value detected for fs.azure.account.key

JVM stacktrace:
com.databricks.common.filesystem.SparkKeyProviderException
	at com.databricks.common.filesystem.LokiABFS.initialize(LokiABFS.scala:48)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$3(LokiFileSystem.scala:137)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$2(LokiFileSystem.scala:133)
	at com.databricks.sql.io.FileSystemCache.getOrCompute(FileSystemCache.scala:60)
	at com.databricks.sql.io.LokiFileSystem$.$anonfun$getLokiFS$1(LokiF

### Métrica 5 · % de viajes con clima adverso

In [0]:
import pyspark.sql.functions as F

df_m5 = df_viajes_clima
if "clima_adverso" not in df_m5.columns:
    col_clima = "clima_codigo" if "clima_codigo" in df_m5.columns else "weather_code"
    df_m5 = df_m5.withColumn("clima_adverso", F.when(F.col(col_clima) > 0, True).otherwise(False))

# M5 · Porcentaje de viajes con clima adverso
m5 = (df_m5
      .filter(F.col("clima_adverso").isNotNull())
      .agg(F.count("*").alias("viajes_con_clima"),
           F.sum(F.col("clima_adverso").cast("int")).alias("viajes_clima_adverso"))
      .withColumn("pct_clima_adverso",
                  F.round(100.0 * F.col("viajes_clima_adverso") / F.col("viajes_con_clima"), 2)))

display(m5) if "display" in dir() else m5.show()

# Análisis por tipo de clima adverso
m5b = (df_m5.filter(F.col("clima_adverso").isNotNull())
       .groupBy("clima_adverso")
       .agg(F.count("*").alias("viajes"),
            F.round(F.avg("duracion_s") / 60, 1).alias("dur_media_min"))
       .orderBy("clima_adverso"))

display(m5b) if "display" in dir() else m5b.show()

viajes_con_clima,viajes_clima_adverso,pct_clima_adverso
244955,163206,66.63


clima_adverso,viajes,dur_media_min
false,81749,5.2
true,163206,5.2


## 8 · Validación de contrato (data quality como código)

In [0]:
def check(nombre, condicion, detalle=""):
    estado = "PASS" if condicion else "FAIL"
    print(f"[{estado}] {nombre} {detalle}")
    return condicion

total = df_viajes_clima.count()

resultados = [
    check("silver_viajes_clima no vacía", total > 0, f"({total:,} filas)"),
    check("sin ts_inicio nulo", 
          df_viajes_clima.filter(F.col("ts_inicio").isNull()).count() == 0),
    check("duracion_s dentro de rango [30, 14400]", 
          df_viajes_clima.filter(~F.col("duracion_s").between(30, 14400)).count() == 0),
    # ... mantén el resto de tus validaciones tal cual las tienes ...
]

[PASS] silver_viajes_clima no vacía (244,955 filas)
[PASS] sin ts_inicio nulo 
[PASS] duracion_s dentro de rango [30, 14400] 


## 9 · Resumen de la corrida (para pegar en el README / evidence)

In [0]:
total = df_viajes_clima.count() if 'df_viajes_clima' in locals() else df_viajes_ok.count()

print(f"""
========================================
NexoUrbano · M3 — resumen de ingesta
========================================
Ejecutado   : {datetime.now().isoformat(timespec='seconds')}
Ciudad      : {P['ciudad']} | Ventana: {P['fecha_inicio']} .. {P['fecha_fin']}
----------------------------------------
Bronze viajes       : {n_filas:,} filas
Silver viajes       : {total:,} filas
Clima Open-Meteo    : {df_clima.count():,} registros horarios
""")


NexoUrbano · M3 — resumen de ingesta
Ejecutado   : 2026-09-26T00:06:00
Ciudad      : Mendoza | Ventana: 2025-03-01 .. 2025-03-31
----------------------------------------
Bronze viajes       : 250,000 filas
Silver viajes       : 244,955 filas
Clima Open-Meteo    : 2,232 registros horarios

